# 09 — Nos données terrain Hanoï

**Le maillon entre la collecte (ODK/Kobo) et le modèle (notebook 07).**

Ce notebook fait tout le travail sur NOS données :
1. Charge l'export CSV de KoboToolbox
2. Nettoie (même pipeline que le notebook 02 sur Sunbird)
3. Enrichit avec la météo (API Open-Meteo, gratuite)
4. Analyses de la Phase 3 du plan : time-series par site, transport vs chantier, dépassements de normes
5. Sauvegarde `data/raw/hanoi/measurements.csv` → consommé par le notebook 07

**Comment exporter depuis Kobo :** ton projet → Data → Downloads → CSV → place le fichier dans `data/raw/hanoi/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import glob

# Le CSV exporté de Kobo (adapte le nom si besoin)
files = glob.glob('../data/raw/hanoi/*.csv')
print('Fichiers trouvés :', files)
raw = pd.read_csv(files[0], sep=';')  # Kobo exporte souvent en ';' — mets ',' si erreur
print(f'{len(raw)} soumissions')
raw.head(3)

In [ ]:
# ---- Mise au format standard ----
# Kobo nomme les colonnes d'après le formulaire. Le geopoint 'location'
# sort en 4 colonnes : _location_latitude, _location_longitude, _location_altitude, _location_precision
# Vérifie les noms exacts avec raw.columns si ça ne matche pas.

df = pd.DataFrame({
    'timestamp':  pd.to_datetime(raw['start_time']),
    'latitude':   raw['_location_latitude'],
    'longitude':  raw['_location_longitude'],
    'altitude':   raw.get('_location_altitude'),
    'accuracy':   raw.get('_location_precision'),
    'noise_dB':   raw['noise_db'],
    'class':      raw['noise_class'],
    'site':       raw['site'],
    'collector':  raw['collector'],
    'dist_to_road': raw.get('dist_to_road'),
    'note':       raw.get('note'),
})

# ---- Nettoyage : même pipeline que le notebook 02 sur Sunbird ----
n0 = len(df)
df = df.dropna(subset=['noise_dB', 'latitude', 'longitude'])
df = df.drop_duplicates(subset=['collector', 'timestamp'])
df = df[df['accuracy'].isna() | (df['accuracy'] < 50)]
df = df[(df['noise_dB'] >= 20) & (df['noise_dB'] <= 120)]

# Correction de calibration croisée entre vos 2 téléphones
# (issue de field/calibration.csv — mets le vrai offset mesuré)
CALIBRATION_OFFSET = {'collector_1': 0.0, 'collector_2': 0.0}
df['noise_dB'] = df['noise_dB'] + df['collector'].map(CALIBRATION_OFFSET).fillna(0)

df['hour'] = df['timestamp'].dt.hour
df['is_weekend'] = df['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
df['day_of_week'] = df['timestamp'].dt.day_name()
print(f'{n0} brutes → {len(df)} mesures propres')

## Météo — API Open-Meteo (gratuite, sans clé)

Le plan demande les conditions météo par mesure. On les récupère depuis les timestamps.

In [ ]:
def get_weather(lat, lon, date_str):
    """Météo horaire pour un jour donné — un appel par jour/site suffit."""
    r = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
        'latitude': lat, 'longitude': lon,
        'start_date': date_str, 'end_date': date_str,
        'hourly': 'temperature_2m,wind_speed_10m,precipitation,relative_humidity_2m',
        'timezone': 'Asia/Bangkok',
    }, timeout=30)
    h = r.json()['hourly']
    return pd.DataFrame(h).assign(time=lambda x: pd.to_datetime(x['time']))

# Un appel par (jour, site) puis merge sur l'heure la plus proche
weather_frames = []
for (date, site), g in df.groupby([df['timestamp'].dt.date, 'site']):
    w = get_weather(g['latitude'].mean(), g['longitude'].mean(), str(date))
    w['site'] = site
    weather_frames.append(w)
weather = pd.concat(weather_frames)

df['time_h'] = df['timestamp'].dt.floor('h')
df = df.merge(
    weather.rename(columns={'time': 'time_h'}),
    on=['time_h', 'site'], how='left'
).drop(columns='time_h')
print('Météo ajoutée :', [c for c in ['temperature_2m','wind_speed_10m','precipitation'] if c in df.columns])

## Phase 3 du plan — Analyses

### Normes utilisées
| Norme | Jour (6h-21h) | Nuit (21h-6h) |
|---|---|---|
| **QCVN 26:2010/BTNMT** (Vietnam, zone ordinaire) | 70 dB | 55 dB |
| **OMS** (recommandation trafic) | 53 dB | 45 dB |

In [ ]:
# Time-series par site : cycle horaire (médiane + IQR, comme la Fig. 9 du papier)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for site, g in df.groupby('site'):
    h = g.groupby('hour')['noise_dB'].median()
    axes[0].plot(h.index, h.values, marker='o', label=site)
axes[0].axhline(70, color='red', linestyle='--', alpha=0.7, label='QCVN jour (70 dB)')
axes[0].axhline(55, color='darkred', linestyle=':', alpha=0.7, label='QCVN nuit (55 dB)')
axes[0].set_xlabel('Heure'); axes[0].set_ylabel('dB (médiane)')
axes[0].set_title('Cycle horaire par site vs normes')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Transport vs construction (demandé par le plan)
df['noise_type'] = np.where(df['class'] == 'construction_site', 'Construction', 'Transportation/other')
sns.boxplot(data=df, x='noise_type', y='noise_dB', hue='is_weekend', ax=axes[1])
axes[1].axhline(70, color='red', linestyle='--', alpha=0.7)
axes[1].set_title('Transport vs Construction (0=semaine, 1=weekend)')

plt.tight_layout()
plt.savefig('../outputs/maps/hanoi_phase3_analysis.png', dpi=150)
plt.show()

In [ ]:
# Dépassements de normes : fréquence et sévérité (demandé par le plan)
df['period'] = np.where((df['hour'] >= 21) | (df['hour'] < 6), 'night', 'day')
df['qcvn_limit'] = np.where(df['period'] == 'day', 70, 55)
df['exceeds_qcvn'] = df['noise_dB'] > df['qcvn_limit']
df['exceed_by'] = (df['noise_dB'] - df['qcvn_limit']).clip(lower=0)

summary = df.groupby(['site', 'period']).agg(
    n=('noise_dB', 'count'),
    median_dB=('noise_dB', 'median'),
    pct_exceed=('exceeds_qcvn', lambda x: 100 * x.mean()),
    mean_exceed_dB=('exceed_by', lambda x: x[x > 0].mean()),
).round(1)
print('=== Dépassements QCVN 26:2010/BTNMT par site ===')
print(summary)
summary.to_csv('../outputs/hanoi_exceedances.csv')

In [ ]:
# ---- Sauvegarde finale : input du notebook 07 (calibration du modèle) ----
out = df[['latitude', 'longitude', 'noise_dB', 'timestamp', 'class', 'site',
          'hour', 'is_weekend'] +
         [c for c in ['temperature_2m', 'wind_speed_10m', 'precipitation'] if c in df.columns]]
out.to_csv('../data/raw/hanoi/measurements.csv', index=False)
print(f'{len(out)} mesures → data/raw/hanoi/measurements.csv')
print('Prochaine étape : notebook 07 (la cellule calibration se décommente et tourne)')